# ENVIRONMENT

In [ ]:
! pip install langchain_community tiktoken langchain-openai langchainhub chromadb langchain

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGCHAIN_API_KEY'] = os.getenv("LANGCHAIN_API_KEY")
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_API_BASE'] = 'https://openrouter.ai/api/v1'
os.environ['OPENAI_BASE_URL'] = 'https://openrouter.ai/api/v1'

In [ ]:
import bs4

from langchainhub import Client

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader

from langchain_chroma import Chroma

from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
loader = WebBaseLoader(
    web_paths = ("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs = dict(
        parse_only = bs4.SoupStrainer(
            class_ = ("post-content" , "post_title" , "post-header")
        )
    ),
)

docs = loader.load()


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

split = text_splitter.split_documents(docs)

In [ ]:
vectorstore = Chroma.from_documents(
    documents=split,
    embedding=OpenAIEmbeddings()
)
retriever = vectorstore.as_retriever()

# INDEX

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

template = """You are an AI language model assistant. Your task is to generate five 
different versions of the given user question to retrieve relevant documents from a vector 
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search. 
Provide these alternative questions separated by newlines. Original question: {question}"""
prompt_perspectives = ChatPromptTemplate.from_template(template)

from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

generate_queries = (
    prompt_perspectives|ChatOpenAI(temperature=0)|StrOutputParser()|(lambda x: x.split("\n"))
)

This creates multiple queries from all perspectives for a single query or question.

In [ ]:
from langchain_core.load import dumps,loads

def get_unique_union(documents: list[list]):
    """ Unique union of retrieved docs """
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]
    unique_docs = list(set(flattened_docs))
    return [loads(doc) for doc in unique_docs]

question = "What is task decomposition for LLM agents?"
retrieval_chain = generate_queries | retriever.map() | get_unique_union
docs = retrieval_chain.invoke({"question":question})
len(docs)

After creating multiple queries, we retrieve a set of chunks for each query and then find the unique chunks across all queries.

In [ ]:
from operator import itemgetter
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough

template = """Answer the following question based on this context:
{context}
Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

llm = ChatOpenAI(temperature=0)

final_rag_chain = (
    {"context": retrieval_chain, 
     "question": itemgetter("question")} 
    | prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"question":question})

This is the final stage where we connect everything together to get the response.

# RAG - FUSION

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

template = """You are a helpful assistant that generates multiple search queries based on a single input query. \n
Generate multiple search queries related to: {question} \n
Output (4 queries):"""
prompt_rag_fusion = ChatPromptTemplate.from_template(template)

from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

generate_queries = (
    prompt_rag_fusion 
    | ChatOpenAI(temperature=0)
    | StrOutputParser() 
    | (lambda x: x.split("\n"))
)

This creates multiple queries from all perspectives for a single query or question.

In [ ]:
from langchain_core.load import dumps,loads

def reciprocal_rank_fusion(results: list[list], k=60):
    """ Reciprocal_rank_fusion that takes multiple lists of ranked documents 
        and an optional parameter k used in the RRF formula """
    
    fused_scores = {}

    for docs in results:
        for rank, doc in enumerate(docs):
            doc_str = dumps(doc)
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            previous_score = fused_scores[doc_str]
            fused_scores[doc_str] += 1 / (rank + k)

    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]

    return reranked_results

retrieval_chain_rag_fusion = generate_queries | retriever.map() | reciprocal_rank_fusion
docs = retrieval_chain_rag_fusion.invoke({"question": question})
len(docs)

In this step, we have created a function that arranges the chunks from most frequent to least frequent based on their number of occurrences.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    {"context": retrieval_chain_rag_fusion, 
     "question": itemgetter("question")} 
    | prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"question":question})

This is the final step where we connect everything and get the response.

### How RAG-Fusion Works

```
┌────────────────────────────────────────────────────────────────────────┐
│                         RAG-FUSION WORKFLOW                            │
└────────────────────────────────────────────────────────────────────────┘

  ┌──────────────────────┐
  │ User asks 1 question │
  └──────────┬───────────┘
             │
             ▼
  ┌──────────────────────────────────┐
  │ LLM rewrites it 4 different ways │
  └───┬──────┬──────┬──────┬─────────┘
      │      │      │      │
      ▼      ▼      ▼      ▼
    ┌───┐  ┌───┐  ┌───┐  ┌───┐
    │ Q1│  │ Q2│  │ Q3│  │ Q4│  Each hits retriever
    └─┬─┘  └─┬─┘  └─┬─┘  └─┬─┘
      │      │      │      │
      └──────┴──────┴──────┘
             │
             ▼
  ┌──────────────────────────────────────┐
  │ RRF scores each chunk by position    │
  │ Chunks ranked highest to lowest      │
  └──────────────────┬───────────────────┘
                     │
                     ▼
  ┌──────────────────────────────────────┐
  │ Top ranked chunks + original Q ─► LLM│
  └──────────────────┬───────────────────┘
                     │
                     ▼
  ┌───────────────────────┐
  │     Final Answer      │
  └───────────────────────┘
```

**Key Insight:**
- Multiple query variations overcome limitations of single-query similarity search
- **RRF (Reciprocal Rank Fusion)** re-ranks chunks by frequency across all queries
- Chunks appearing in multiple result sets get higher scores

# DECOMPOSION

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

template = """You are a helpful assistant that generates multiple sub-questions related to an input question. \n
The goal is to break down the input into a set of sub-problems / sub-questions that can be answers in isolation. \n
Generate multiple search queries related to: {question} \n
Output (3 queries):"""
prompt_decomposition = ChatPromptTemplate.from_template(template)

In this step, we created a template that can generate sub-queries from a single query to get the best response.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(temperature=0)

generate_queries_decomposition = ( prompt_decomposition | llm | StrOutputParser() | (lambda x: x.split("\n")))

question = "What are the main components of an LLM-powered autonomous agent system?"
questions = generate_queries_decomposition.invoke({"question":question})

In this block of code, we generate sub-queries from a single query.

In this step, we get the answer to each query individually.

In [ ]:
answers = []
for q in questions:
    if q.strip():
        clean_q = q
        for prefix in [". ", ") "]:
            if prefix in q[:5]:
                clean_q = q.split(prefix, 1)[-1]
                break
        sub_prompt = ChatPromptTemplate.from_template(
            "Answer the following question based on this context:\n{context}\nQuestion: {question}"
        )
        sub_rag_chain = (
            {"context": retriever, "question": RunnablePassthrough()}
            | sub_prompt
            | llm
            | StrOutputParser()
        )
        answers.append(sub_rag_chain.invoke(clean_q))

def format_qa_pairs(questions, answers):
    """Format Q and A pairs"""
    formatted_string = ""
    for i, (question, answer) in enumerate(zip(questions, answers), start=1):
        formatted_string += f"Question {i}: {question}\nAnswer {i}: {answer}\n\n"
    return formatted_string.strip()

context = format_qa_pairs(questions, answers)
template = """Here is a set of Q+A pairs:

{context}

Use these to synthesize an answer to the question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"context":context,"question":question})

This is the final step, where we connect everything and get the final response (using the individual approach).

### How Decomposition (Individual/Parallel) Works

```
┌─────────────────────────────────────────────────────────────────────────┐
│              DECOMPOSITION (INDIVIDUAL / PARALLEL)                      │
└─────────────────────────────────────────────────────────────────────────┘

  ┌───────────────────────────┐
  │ User asks 1 big question  │
  └─────────────┬─────────────┘
                │
                ▼
  ┌──────────────────────────────────────┐
  │   LLM breaks into 3 sub-questions    │
  └────┬─────────────┬─────────────┬─────┘
       │             │             │
       ▼             ▼             ▼
  ┌─────────┐   ┌─────────┐   ┌─────────┐
  │   Q1    │   │   Q2    │   │   Q3    │
  │retrieve │   │retrieve │   │retrieve │
  │ answer1 │   │ answer2 │   │ answer3 │
  └────┬────┘   └────┬────┘   └────┬────┘
       │             │             │
       └─────────────┴─────────────┘
                     │
                     ▼
  ┌──────────────────────────────────────┐
  │        All Q&A pairs combined        │
  │ Combined context + original Q ─► LLM │
  └──────────────────┬───────────────────┘
                     │
                     ▼
         ┌───────────────────────┐
         │     Final Answer      │
         └───────────────────────┘
```

**Key Insight:**
- LLM breaks a complex query into independent sub-questions
- Sub-questions are executed **in parallel** (saving time)
- Useful when sub-questions do not depend on each other's answers

In [ ]:
template = """Here is the question you need to answer:

\n --- \n {question} \n --- \n

Here is any available background question + answer pairs:

\n --- \n {q_a_pairs} \n --- \n

Here is additional context relevant to the question: 

\n --- \n {context} \n --- \n

Use the above context and any background question + answer pairs to answer the question: \n {question}
"""

decomposition_prompt = ChatPromptTemplate.from_template(template)

We create a template so that we can pass the reference of the previous question-answer pair to the next one.

In [ ]:
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

def format_qa_pair(question, answer):
    """Format Q and A pair"""
    
    formatted_string = ""
    formatted_string += f"Question: {question}\nAnswer: {answer}\n\n"
    return formatted_string.strip()

# llm
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

q_a_pairs = ""
for q in questions:
    
    rag_chain = (
    {"context": itemgetter("question") | retriever, 
     "question": itemgetter("question"),
     "q_a_pairs": itemgetter("q_a_pairs")} 
    | decomposition_prompt
    | llm
    | StrOutputParser())

    answer = rag_chain.invoke({"question":q,"q_a_pairs":q_a_pairs})
    q_a_pair = format_qa_pair(q,answer)
    q_a_pairs = q_a_pairs + "\n---\n"+  q_a_pair

This is the final stage where we connect everything and get the final response (using the recursive approach).

### How Decomposition (Recursive/Sequential) Works

```
┌─────────────────────────────────────────────────────────────────────────┐
│            DECOMPOSITION (RECURSIVE / SEQUENTIAL)                       │
└─────────────────────────────────────────────────────────────────────────┘

  ┌────────────────────────────┐
  │  User asks 1 big question  │
  └────────────┬───────────────┘
               │
               ▼
  ┌──────────────────────────────────┐
  │  LLM breaks into 3 sub-questions │
  └────────────────┬─────────────────┘
                   │
                   ▼
  ┌─────────────────────────────────────────────────┐
  │  Q1 ─► retrieve chunks ─► LLM answers Q1        │
  └──────────────────────┬──────────────────────────┘
                         │
                         ▼  passes Answer1 forward
  ┌─────────────────────────────────────────────────┐
  │  Q2 + Answer1 ─► retrieve ─► LLM answers Q2     │
  └──────────────────────┬──────────────────────────┘
                         │
                         ▼  passes Answer1+2 forward
  ┌─────────────────────────────────────────────────┐
  │  Q3 + Answer1+2 ─► retrieve ─► LLM answers Q3   │
  └──────────────────────┬──────────────────────────┘
                         │
                         ▼
            ┌───────────────────────┐
            │     Final Answer      │
            └───────────────────────┘
```

**Key Insight:**
- Each sub-question builds on the **previous answer** (sequential chain)
- Prior Q&A context is passed forward, enabling **deeper reasoning**
- Works well when sub-questions are **dependent** on each other

# STEP - BACK

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
examples = [
    {
        "input": "Could the members of The Police perform lawful arrests?",
        "output": "what can the members of The Police do?",
    },
    {
        "input": "Jan Sindel's was born in what country?",
        "output": "what is Jan Sindel's personal history?",
    },
]
# We now transform these to example messages
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are an expert at world knowledge. Your task is to step back and paraphrase a question to a more generic step-back question, which is easier to answer. Here are a few examples:""",
        ),
        # Few shot examples
        few_shot_prompt,
        # New question
        ("user", "{question}"),
    ]
)

Here, we use few-shot prompting. In this step, we convert the query into a more generic and relevant one to improve the chance of matching chunks.

In [ ]:
generate_queries_step_back = prompt | ChatOpenAI(temperature=0) | StrOutputParser()
question = "What is task decomposition for LLM agents?"
generate_queries_step_back.invoke({"question": question})

Here, we pass the question along with the prompt to the OpenAI model.

In [ ]:
from langchain_core.runnables import RunnableLambda

response_prompt_template = """You are an expert of world knowledge. I am going to ask you a question. Your response should be comprehensive and not contradicted with the following context if they are relevant. Otherwise, ignore them if they are not relevant.

# {normal_context}
# {step_back_context}

# Original Question: {question}
# Answer:"""
response_prompt = ChatPromptTemplate.from_template(response_prompt_template)

chain = (
    {
        "normal_context": RunnableLambda(lambda x: x["question"]) | retriever,
        "step_back_context": generate_queries_step_back | retriever,
        "question": lambda x: x["question"],
    }
    | response_prompt
    | ChatOpenAI(temperature=0)
    | StrOutputParser()
)

chain.invoke({"question": question})

This is the final stage where we provide the normal query context, the new query context, and the question, connecting all of these to get the final response.

### How Step-Back Works

```
┌─────────────────────────────────────────────────────────────────────────┐
│                       STEP-BACK WORKFLOW                                │
└─────────────────────────────────────────────────────────────────────────┘

  ┌─────────────────────────────┐
  │ User asks specific question │
  └──────────────┬──────────────┘
                 │
                 ▼
 ┌────────────────────────────────────────┐
 │ LLM makes it more generic              │
 │ (step-back question)                   │
 └───────────────────┬────────────────────┘
                     │
          ┌──────────┴─────────┐
          │                    │
          ▼                    ▼
  ┌────────────────┐  ┌──────────────────┐
  │ Original Q     │  │ Step-back Q      │
  │ ─► retriever   │  │ ─► retriever     │
  │ ─► normal      │  │ ─► step-back     │
  │    context     │  │    context       │
  └───────┬────────┘  └────────┬─────────┘
          │                    │
          └──────────┬─────────┘
                     │
                     ▼
 ┌───────────────────────────────────────────┐
 │ Both contexts + original question ─► LLM  │
 └────────────────────┬──────────────────────┘
                      │
                      ▼
          ┌───────────────────────┐
          │     Final Answer      │
          └───────────────────────┘
```

**Key Insight:**
- The step-back question is a **more generic version** of the original
- Retrieves **broader context** that may contain the specific answer
- Combines both normal and step-back context for a comprehensive response

# HYDE

In [ ]:
from langchain_core.prompts import ChatMessagePromptTemplate

template = """Please write a scientific paper passage to answer the question
Question: {question}
Passage:"""
prompt_hyde = ChatPromptTemplate.from_template(template)

from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

generate_docs_for_retrieval = (
    prompt_hyde | ChatOpenAI(temperature=0) | StrOutputParser() 
)

question = "What is task decomposition for LLM agents?"
generate_docs_for_retrieval.invoke({"question":question})

Here, we generate a hypothetical response from the query and then retrieve the real response based on that hypothetical answer.

In [ ]:
retrieval_chain = generate_docs_for_retrieval | retriever 
retrieved_docs = retrieval_chain.invoke({"question":question})
retrieved_docs

Here, we create a hypothetical response.

In [ ]:
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"context":retrieved_docs,"question":question})

Here, we generate the real response using the hypothetical response.

### How HyDE (Hypothetical Document Embeddings) Works

```
┌─────────────────────────────────────────────────────────────────────────┐
│         HYDE (HYPOTHETICAL DOCUMENT EMBEDDINGS)                         │
└─────────────────────────────────────────────────────────────────────────┘

  ┌────────────────────────────────────────┐
  │  User asks question                    │
  └──────────────────┬─────────────────────┘
                     │
                     ▼
  ┌────────────────────────────────────────┐
  │  LLM generates fake detailed answer    │
  │  (hypothetical passage)                │
  └──────────────────┬─────────────────────┘
                     │
                     ▼
  ┌────────────────────────────────────────┐
  │  Fake answer ─► embedded ─► search     │
  │  Chroma (similarity search)            │
  └──────────────────┬─────────────────────┘
                     │
                     ▼
  ┌────────────────────────────────────────┐
  │  Real matching chunks retrieved        │
  └──────────────────┬─────────────────────┘
                     │
                     ▼
  ┌────────────────────────────────────────┐
  │  Real chunks + original question ─► LLM│
  └──────────────────┬─────────────────────┘
                     │
                     ▼
         ┌───────────────────────┐
         │     Final Answer      │
         └───────────────────────┘
```

**Key Insight:**
- The fake answer is **closer in embedding space** to real documents than the question itself
- This bridges the gap between question-style and document-style text
- Retrieval quality improves because the search uses document-like embeddings